In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number, when
from pyspark.sql.window import Window
import numpy as np
import pandas as pd
import json

spark = SparkSession.builder \
    .appName("Pertemuan7-StudiKasusTerintegrasi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [4]:
np.random.seed(77)

# SUMBER 1: Data toko/cabang (JSON) — tabel referensi kecil
toko_list = ["Toko Magelang", "Toko Yogyakarta", "Toko Semarang", "Toko Solo"]
data_toko = [
    {"toko_id": i + 1, "nama_toko": t, "kota": t.split()[1], "manager": np.random.choice(["Rani", "Joko", "Sari", "Bayu"])}
    for i, t in enumerate(toko_list)
]
with open("data_toko.json", "w") as f:
    for t in data_toko:
        f.write(json.dumps(t) + "\n")

# SUMBER 2: Data produk (CSV) — tabel referensi kecil
data_produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}",
     "kategori": np.random.choice(["Elektronik", "Fashion", "Makanan"]),
     "harga": int(np.random.choice([50000, 100000, 200000]))}
    for i in range(1, 21)
]
pd.DataFrame(data_produk).to_csv("data_produk.csv", index=False)

# SUMBER 3: Data transaksi triwulan (CSV) — data utama, jauh lebih besar
n = 5000
data_transaksi = pd.DataFrame({
    "trx_id": [f"T{i}" for i in range(n)],
    "toko_id": np.random.randint(1, 5, size=n),
    "product_id": np.random.randint(1, 21, size=n),
    "qty": np.random.randint(1, 6, size=n),
})
data_transaksi.to_csv("data_transaksi_triwulan.csv", index=False)

print("Ketiga sumber data berhasil dibuat:")
print(f"- data_toko.json              : {len(data_toko)} baris")
print(f"- data_produk.csv             : {len(data_produk)} baris")
print(f"- data_transaksi_triwulan.csv : {len(data_transaksi)} baris")

Ketiga sumber data berhasil dibuat:
- data_toko.json              : 4 baris
- data_produk.csv             : 20 baris
- data_transaksi_triwulan.csv : 5000 baris


In [5]:
!hdfs dfs -mkdir -p /user/mahasiswa/pertemuan7/raw
!hdfs dfs -put -f data_transaksi_triwulan.csv /user/mahasiswa/pertemuan7/raw/
!hdfs dfs -put -f data_toko.json /user/mahasiswa/pertemuan7/raw/
!hdfs dfs -put -f data_produk.csv /user/mahasiswa/pertemuan7/raw/
!hdfs dfs -ls /user/mahasiswa/pertemuan7/raw

Found 3 items
-rw-r--r--   1 raka supergroup        599 2026-10-04 10:53 /user/mahasiswa/pertemuan7/raw/data_produk.csv
-rw-r--r--   1 raka supergroup        332 2026-10-04 10:53 /user/mahasiswa/pertemuan7/raw/data_toko.json
-rw-r--r--   1 raka supergroup      61655 2026-10-04 10:53 /user/mahasiswa/pertemuan7/raw/data_transaksi_triwulan.csv


In [7]:
df_transaksi = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/pertemuan7/raw/data_transaksi_triwulan.csv",
    header=True, inferSchema=True
)
df_toko = spark.read.json("hdfs://localhost:9000/user/mahasiswa/pertemuan7/raw/data_toko.json")
df_produk = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/pertemuan7/raw/data_produk.csv",
    header=True, inferSchema=True
)

print("Transaksi:", df_transaksi.count(), "| Toko:", df_toko.count(), "| Produk:", df_produk.count())

Transaksi: 5000 | Toko: 4 | Produk: 20


In [9]:
df_gabungan = df_transaksi.join(df_toko, on="toko_id", how="inner") \
                           .join(df_produk, on="product_id", how="inner")

df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))
df_gabungan.show(5)

+----------+-------+------+---+----------+-------+---------------+-----------+----------+------+---------------+
|product_id|toko_id|trx_id|qty|      kota|manager|      nama_toko|nama_produk|  kategori| harga|total_penjualan|
+----------+-------+------+---+----------+-------+---------------+-----------+----------+------+---------------+
|         7|      1|    T0|  5|  Magelang|   Bayu|  Toko Magelang|   Produk-7|   Makanan|200000|        1000000|
|         5|      2|    T1|  5|Yogyakarta|   Bayu|Toko Yogyakarta|   Produk-5|Elektronik|200000|        1000000|
|         2|      3|    T2|  2|  Semarang|   Rani|  Toko Semarang|   Produk-2|Elektronik|100000|         200000|
|         2|      2|    T3|  2|Yogyakarta|   Bayu|Toko Yogyakarta|   Produk-2|Elektronik|100000|         200000|
|        18|      4|    T4|  2|      Solo|   Rani|      Toko Solo|  Produk-18|Elektronik|100000|         200000|
+----------+-------+------+---+----------+-------+---------------+-----------+----------+------+

In [13]:
# Meringkas total penjualan per kombinasi toko + produk terlebih dahulu
ringkasan_produk_toko = df_gabungan.groupBy("nama_toko", "nama_produk").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

# Menerapkan window function untuk mengambil top-3 produk pada setiap toko
window_toko = Window.partitionBy("nama_toko").orderBy(col("total_penjualan").desc())
top3_produk = ringkasan_produk_toko.withColumn("peringkat", row_number().over(window_toko)) \
                                    .filter(col("peringkat") <= 3)

top3_produk.orderBy("nama_toko", "peringkat").show(20)

+---------------+-----------+---------------+---------+
|      nama_toko|nama_produk|total_penjualan|peringkat|
+---------------+-----------+---------------+---------+
|  Toko Magelang|   Produk-7|       46000000|        1|
|  Toko Magelang|   Produk-5|       44400000|        2|
|  Toko Magelang|  Produk-14|       38400000|        3|
|  Toko Semarang|   Produk-8|       49200000|        1|
|  Toko Semarang|   Produk-7|       40400000|        2|
|  Toko Semarang|   Produk-5|       39000000|        3|
|      Toko Solo|  Produk-14|       38400000|        1|
|      Toko Solo|   Produk-7|       37200000|        2|
|      Toko Solo|   Produk-5|       33200000|        3|
|Toko Yogyakarta|  Produk-14|       48200000|        1|
|Toko Yogyakarta|   Produk-5|       37600000|        2|
|Toko Yogyakarta|   Produk-4|       37000000|        3|
+---------------+-----------+---------------+---------+



In [15]:
df_gabungan.createOrReplaceTempView("gabungan")

kontribusi_manager = spark.sql('''
    SELECT kota, manager, SUM(total_penjualan) AS total_penjualan
    FROM gabungan
    GROUP BY kota, manager
    ORDER BY total_penjualan DESC
''')
kontribusi_manager.show()

+----------+-------+---------------+
|      kota|manager|total_penjualan|
+----------+-------+---------------+
|  Semarang|   Rani|      416250000|
|  Magelang|   Bayu|      408550000|
|Yogyakarta|   Bayu|      389150000|
|      Solo|   Rani|      369950000|
+----------+-------+---------------+



In [16]:
!hdfs dfs -mkdir -p /user/mahasiswa/pertemuan7/processed

df_gabungan.write.mode("overwrite").partitionBy("kategori").parquet(
    "hdfs://localhost:9000/user/mahasiswa/pertemuan7/processed/laporan_triwulan"
)

print("Laporan berhasil disimpan ke HDFS dalam format Parquet, terpartisi per kategori produk.")
!hdfs dfs -ls /user/mahasiswa/pertemuan7/processed/laporan_triwulan

Laporan berhasil disimpan ke HDFS dalam format Parquet, terpartisi per kategori produk.
Found 4 items
-rw-r--r--   3 raka supergroup          0 2026-10-04 11:01 /user/mahasiswa/pertemuan7/processed/laporan_triwulan/_SUCCESS
drwxr-xr-x   - raka supergroup          0 2026-10-04 11:01 /user/mahasiswa/pertemuan7/processed/laporan_triwulan/kategori=Elektronik
drwxr-xr-x   - raka supergroup          0 2026-10-04 11:01 /user/mahasiswa/pertemuan7/processed/laporan_triwulan/kategori=Fashion
drwxr-xr-x   - raka supergroup          0 2026-10-04 11:01 /user/mahasiswa/pertemuan7/processed/laporan_triwulan/kategori=Makanan


In [18]:
df_final = spark.read.parquet("hdfs://localhost:9000/user/mahasiswa/pertemuan7/processed/laporan_triwulan")

ringkasan_eksekutif = df_final.groupBy("nama_toko", "kota", "manager").agg(
    spark_sum("total_penjualan").alias("total_penjualan_triwulan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan_triwulan").desc())

ringkasan_eksekutif.show()

[Stage 84:=============================>                            (1 + 1) / 2]

+---------------+----------+-------+------------------------+----------------+-------------------------+
|      nama_toko|      kota|manager|total_penjualan_triwulan|jumlah_transaksi|rata_rata_nilai_transaksi|
+---------------+----------+-------+------------------------+----------------+-------------------------+
|  Toko Semarang|  Semarang|   Rani|               416250000|            1306|       318721.28637059726|
|  Toko Magelang|  Magelang|   Bayu|               408550000|            1315|       310684.41064638784|
|Toko Yogyakarta|Yogyakarta|   Bayu|               389150000|            1200|        324291.6666666667|
|      Toko Solo|      Solo|   Rani|               369950000|            1179|        313782.8668363019|
+---------------+----------+-------+------------------------+----------------+-------------------------+



In [19]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
